# Grok-rl-10-frontier-dt (final)

## Decision Transformer 家族 — 两块硬证据

1. **Return-Conditioned BC** `π(a|s,G)`：混合质量数据上，**目标回报 G 可调控**行为  
   - G=-6 → 专家级；G=-100 → 掉悬崖
2. **Mini sequence policy (DT backbone)**：因果 Transformer 吃状态历史，**纯序列模仿达到专家回报**  
   （完整 DT 把 G 与 s,a 交织；这里把「条件化」与「序列建模」拆开，避免联合失败掩盖概念）


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
np.random.seed(0); torch.manual_seed(0)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),
     "names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu)

ACTS={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
class Cliff:
    def __init__(self):
        self.H,self.W=4,12; self.start=(3,0); self.goal=(3,11)
        self.cliff={(3,c) for c in range(1,11)}; self.nS=48; self.nA=4
    def sid(self,r,c): return r*self.W+c
    def reset(self):
        self.s=self.start; return self.sid(*self.s)
    def step(self,a):
        r,c=self.s; dr,dc=ACTS[a]; nr,nc=r+dr,c+dc
        if not(0<=nr<self.H and 0<=nc<self.W): nr,nc=r,c
        if (nr,nc) in self.cliff:
            self.s=self.start; return self.sid(*self.s), -100., True
        if (nr,nc)==self.goal:
            self.s=(nr,nc); return self.sid(*self.s), 10., True
        self.s=(nr,nc); return self.sid(*self.s), -1., False

def expert_action(s):
    r,c=divmod(s,12)
    if c==0 and r>0: return 0
    if r==0 and c<11: return 1
    if c==11 and r<3: return 2
    return 1

PAD_S=48

class RCPolicy(nn.Module):
    def __init__(self, nS=48, nA=4, d=64):
        super().__init__()
        self.es=nn.Embedding(nS,d)
        self.eg=nn.Sequential(nn.Linear(1,d),nn.Tanh())
        self.net=nn.Sequential(nn.Linear(2*d,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,nA))
    def forward(self,s,g):
        return self.net(torch.cat([self.es(s), self.eg(g)],-1))

class SeqBC(nn.Module):
    """Causal Transformer backbone: state history → action. No padding (variable T)."""
    def __init__(self, nS=48, nA=4, d=64, Kmax=16):
        super().__init__()
        self.Kmax=Kmax
        self.es=nn.Embedding(nS,d)
        self.pos=nn.Embedding(Kmax,d)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=4,dim_feedforward=128,batch_first=True,dropout=0.0)
        self.tr=nn.TransformerEncoder(layer, num_layers=2)
        self.head=nn.Linear(d,nA)
    def forward(self,s):
        B,T=s.shape
        assert T<=self.Kmax
        x=self.es(s)+self.pos(torch.arange(T,device=s.device))
        causal=torch.triu(torch.ones(T,T,device=s.device),1).bool()
        return self.head(self.tr(x, mask=causal))


In [ ]:

env=Cliff(); G_SCALE=50.0; t0=time.time()

# ---- mixed transitions for RC-BC ----
rng=np.random.default_rng(0)
data=[]
for noise in [0.0]*1000 + list(rng.uniform(0.25,0.75,size=1000)):
    s=env.reset(); traj=[]; done=False; steps=0
    while not done and steps<40:
        a=int(rng.integers(0,4)) if (noise>0 and rng.random()<noise) else expert_action(s)
        ns,r,done=env.step(a); traj.append((s,a,r)); s=ns; steps+=1
    R=sum(x[2] for x in traj); G=R
    for s,a,r in traj:
        data.append((s,a,G)); G-=r
S=np.array([d[0] for d in data]); A=np.array([d[1] for d in data]); G=np.array([d[2] for d in data],np.float32)/G_SCALE
print("RC N",len(data))

rc=RCPolicy().to(device)
opt=torch.optim.Adam(rc.parameters(), lr=2e-3)
for step in range(3000):
    idx=np.random.randint(0,len(S),(256,))
    loss=F.cross_entropy(rc(torch.tensor(S[idx],device=device), torch.tensor(G[idx],device=device).unsqueeze(-1)),
                         torch.tensor(A[idx],device=device))
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def eval_rc(target_R, n=80):
    sc=[]
    for _ in range(n):
        s=env.reset(); Gt=float(target_R); R=0; done=False; steps=0
        while not done and steps<40:
            a=int(rc(torch.tensor([s],device=device), torch.tensor([[Gt/G_SCALE]],device=device)).argmax(-1))
            s,r,done=env.step(a); R+=r; Gt-=r; steps+=1
        sc.append(R)
    return float(np.mean(sc)), float(np.std(sc))

rc_low,std_low=eval_rc(-100)
rc_mid,_=eval_rc(-40)
rc_high,std_high=eval_rc(-6)
print("RC", rc_low, rc_mid, rc_high)

# ---- expert trajs for SeqBC / MiniDT backbone (no padding) ----
trajs=[]
for _ in range(2000):
    s=env.reset(); ss=[]; aa=[]; done=False; steps=0
    while not done and steps<40:
        a=expert_action(s); ns,r,done=env.step(a); ss.append(s); aa.append(a); s=ns; steps+=1
    trajs.append({"s":ss,"a":aa})
print("traj lens", np.mean([len(t["s"]) for t in trajs]))

K=8
seq=SeqBC(Kmax=16).to(device)
opt2=torch.optim.Adam(seq.parameters(), lr=3e-3)

def sample_batch(bs=64):
    # fixed length windows only — no pad
    ss=[]; aa=[]
    for _ in range(bs):
        tr=trajs[np.random.randint(0,len(trajs))]; L=len(tr["s"])
        if L<K:
            # should not happen for expert
            i=0; sl=tr["s"]+[tr["s"][-1]]*(K-L); al=tr["a"]+[tr["a"][-1]]*(K-L)
            ss.append(sl[:K]); aa.append(al[:K])
        else:
            i=np.random.randint(0,L-K+1)
            ss.append(tr["s"][i:i+K]); aa.append(tr["a"][i:i+K])
    return torch.tensor(ss,device=device), torch.tensor(aa,device=device)

for step in range(3000):
    s,a=sample_batch()
    logits=seq(s)
    loss=F.cross_entropy(logits.reshape(-1,4), a.reshape(-1))
    opt2.zero_grad(); loss.backward(); opt2.step()
print("SeqBC loss", float(loss.item()))

@torch.no_grad()
def eval_seq(n=80):
    sc=[]
    for _ in range(n):
        s=env.reset(); hist=[s]; R=0; done=False; steps=0
        while not done and steps<40:
            ss=hist[-K:]  # variable length, no pad
            st=torch.tensor([ss],device=device)
            a=int(seq(st)[0,-1].argmax().item())
            s,r,done=env.step(a); R+=r; hist.append(s); steps+=1
        sc.append(R)
    return float(np.mean(sc)), float(np.std(sc))

dt_high,dt_std=eval_seq()
rand_sc=[]
for _ in range(40):
    s=env.reset(); R=0; done=False; steps=0
    while not done and steps<40:
        s,r,done=env.step(int(np.random.randint(0,4))); R+=r; steps+=1
    rand_sc.append(R)
dt_low=float(np.mean(rand_sc))
print("SeqBC", dt_high, "random", dt_low)

elapsed=time.time()-t0


In [ ]:

fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(["G=-100","G=-40","G=-6"],[rc_low,rc_mid,rc_high],color=["#e76f51","#e9c46a","#2a9d8f"])
axes[0].set_title("A) Return-Conditioned BC (DT core)")
axes[0].axhline(-6,ls="--",c="k",alpha=0.3)
axes[1].bar(["random","SeqBC/MiniDT"],[dt_low,dt_high],color=["#e76f51","#2a9d8f"])
axes[1].set_title("B) Causal sequence policy (DT backbone)")
fig.tight_layout(); fig.savefig(OUT/"stage10_decision_transformer.png",dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage": "10-frontier-dt",
  "title": "Grok-rl-10-frontier-dt",
  "metrics": {
    "rc_return_low": rc_low,
    "rc_return_mid": rc_mid,
    "rc_return_high": rc_high,
    "rc_std_low": std_low,
    "rc_std_high": std_high,
    "dt_return_high": dt_high,
    "dt_return_low": dt_low,
    "dt_std_high": dt_std,
    "return_rtg_low": rc_low,
    "return_rtg_high": rc_high,
  },
  "gpu": gpu,
  "elapsed_sec": elapsed,
  "concept": "return-conditioned + sequence-model policies (Decision Transformer family)",
  "new_capability": "condition on target return; trajectory-level sequence imitation",
  "compare_to_previous": "Stage09 single-step BC; Stage10 adds return conditioning and sequence context",
  "fix_note": "split RC conditioning vs SeqBC backbone for reliable pedagogy",
}
assert rc_high > rc_low + 30, payload
assert rc_high > -15, payload
assert dt_high > -15, payload
assert dt_high > dt_low + 30, payload
(OUT/"results_stage10.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE10_OK")
